# makeshift — quick start

This notebook covers the core workflow:
- Fetching an NMR-STAR file from BMRB
- Parsing it into a tidy chemical shift dataframe
- Handling multi-entity entries
- Calculating and visualising the Chemical Shift Index (CSI)

In [ ]:
import makeshift as ms
import seaborn as sns
import matplotlib.pyplot as plt

## 1. Fetch and parse a BMRB entry

`fetch_nmrstar_file` downloads the NMR-STAR v3 file for a given BMRB ID and saves it locally.
`parse_nmr_star` reads it into a nested dict keyed by saveframe category.

In [ ]:
# Unphosphorylated NtrCr — Volkman & Kern 2001
ms.fetch_nmrstar_file(4527)  # saves bmr4527_3.str to the current directory
entry = ms.parse_nmr_star('bmr4527_3.str')

## 2. Extract chemical shifts

`get_chem_shifts` returns a tidy dataframe with one row per observed chemical shift.

Key columns:
| Column | Description |
|---|---|
| `Seq_ID` | Residue sequence number (1-indexed) |
| `Comp_ID` | Three-letter amino acid code |
| `Atom_ID` | Atom name (CA, CB, N, H, HA, C, ...) |
| `Atom_type` | Element (C, N, H) |
| `Val` | Observed chemical shift (ppm) |
| `cs_saveframe_id` | Which saveframe the shift came from (relevant for multi-saveframe entries) |

In [ ]:
cs = ms.get_chem_shifts(entry)
cs

## 3. Simulate an HSQC

Pivot to one row per residue with N and H columns, then plot.

In [ ]:
pivoted = (
    cs[cs['Atom_ID'].isin(['N', 'H'])]
    .pivot_table(
        index=['Entity_ID', 'Seq_ID', 'Comp_ID'],
        columns='Atom_type',
        values='Val',
    )
    .reset_index()
)
pivoted.columns.name = None

sns.scatterplot(x='H', y='N', data=pivoted)
plt.gca().invert_yaxis()
plt.gca().invert_xaxis()
plt.xlabel('¹H (ppm)')
plt.ylabel('¹⁵N (ppm)')
plt.title('Simulated HSQC — NtrCr (4527)')

## 4. Multi-entity entries

Some BMRB entries contain multiple entities (e.g. a protein + DNA + ligand).
`get_sequences` lists them so you can filter to the one you want.

In [ ]:
# hERR2 zinc finger bound to DNA — four entities: two DNA strands, protein, Zn
ms.fetch_nmrstar_file(5363)
entry = ms.parse_nmr_star('bmr5363_3.str')

entities = ms.get_sequences(entry)
entities

In [ ]:
cs = ms.get_chem_shifts(entry)

# Filter to polypeptide entity only
protein_ids = list(entities.loc[entities['Polymer_type'] == 'polypeptide(L)', 'ID'])
pcs = cs.loc[cs.Entity_ID.isin(protein_ids)]
pcs

## 5. Chemical Shift Index (CSI)

Pass `calc_CSI=True` to compute two additional columns:

| Column | Description |
|---|---|
| `csi_raw` | (CA − CB) secondary shift (observed − random coil). Positive = helix tendency, negative = strand tendency. GLY falls back to CA-only secondary shift since it has no CB. |
| `csi` | Discretised index: +1 (helix), −1 (strand), 0 (coil/ambiguous) |

In [ ]:
# KaiB-TV4 — a well-structured protein with known secondary structure
ms.fetch_nmrstar_file(31107)
entry = ms.parse_nmr_star('bmr31107_3.str')
cs = ms.get_chem_shifts(entry, calc_CSI=True)

# Known sequence and DSSP secondary structure assignment
seq = 'MYVFRLYVRGETHAAEVALKNLHDLLSSALKVPYTLKVVDVTKQPDLAEKDQVQATPTLVRVYPQPVRRLVGQLDHRYRLQHLLSP'
dss = 'CEEEEEEECCCHHHHHHHHHHHHHHHHHHHCCCCEEEEEECCCHHHHHHHHHCCCCCEEEECCCCCCEEEECCCCHHHHHHHHHHC'

tmp = cs.loc[cs.Atom_type == 'N'].copy()
tmp['dssp'] = tmp.apply(lambda row: dss[row['Seq_ID'] - 1], axis=1)

plt.figure(figsize=(10, 2))
sns.scatterplot(x='Seq_ID', y='csi_raw', hue='dssp', data=tmp)
plt.axhline(0, color='k', linewidth=0.5, linestyle='--')
plt.xlabel('Residue')
plt.ylabel('CSI (ppm)')
plt.title('CSI — KaiB-TV4 (31107)')
plt.legend(bbox_to_anchor=(1, 1))

In [ ]:
# Intrinsically disordered protein — CSI fluctuates around zero with no sustained trend
ms.fetch_nmrstar_file(27401)
entry = ms.parse_nmr_star('bmr27401_3.str')
cs = ms.get_chem_shifts(entry, calc_CSI=True)

tmp = cs.loc[cs.Atom_type == 'N']

plt.figure(figsize=(10, 2))
sns.scatterplot(x='Seq_ID', y='csi_raw', data=tmp)
plt.axhline(0, color='k', linewidth=0.5, linestyle='--')
plt.ylim([-6, 6])
plt.xlabel('Residue')
plt.ylabel('CSI (ppm)')
plt.title('CSI — IDP (27401)')